# Pilot Annotation Evaluation

This section evaluates KA and ACC metrics for the pilot annotation phases only. These stages are examined separately, as they were primarily intended to train annotators and refine the annotation guidelines.


In [1]:
import pandas as pd
import glob
import krippendorff
import os

In [2]:
pilot_data = glob.glob('../Datasets/pilot*.csv')

In [3]:
mapping_3class = {
    'Positive': 'Positive',
    'M_Positive': 'Positive',    
    'Negative': 'Negative',
    'M_Negative': 'Negative',
    'P_Neutral': 'Neutral',
    'N_Neutral': 'Neutral',
    }

results = []

for file in pilot_data:
    df = pd.read_csv(file)
    pilot_name = os.path.splitext(os.path.basename(file))[0]

    
    #Selecting columns with annotation tags (tag_tamara, tag_katja, tag_anze)
    tag_cols = [col for col in df.columns if col.startswith("tag_")]

    #Handle values with added Sarcasm note (Negative(S) -> Negative)
    for col in tag_cols:
        df[col] = df[col].astype("str").str.strip().str.replace(r"\(S\)", "", regex=True)

    #---- 6-class KA calculation---#
    unique_tag6 = pd.unique(df[tag_cols].values.ravel())
    tag_to_code6 = {label: idx for idx, label in enumerate(unique_tag6)}

    for col in tag_cols:
        df[col + '_6class'] = df[col].map(tag_to_code6)

    data_6class = df[[col + '_6class' for col in tag_cols]].to_numpy().T
    alpha_6class = krippendorff.alpha(reliability_data=data_6class, level_of_measurement='nominal')

    #---3-class KA calculation--#
    for col in tag_cols:
        df[col + '_3class'] = df[col].map(mapping_3class)

    unique_tag3 = pd.unique(df[[col + "_3class" for col in tag_cols]].values.ravel())
    tag_to_code3 = {label: idx for idx, label in enumerate(unique_tag3)}

    for col in tag_cols:
        df[col + '_3code'] = df[col + '_3class'].map(tag_to_code3)
    
    data_3class = df[[col + '_3code' for col in tag_cols]].to_numpy().T
    alpha_3class = krippendorff.alpha(reliability_data=data_3class, level_of_measurement="nominal")

    #----Percentage of agreed-upon instances---#
    def percent_agreement(rows):
        values = [val for val in rows if pd.notnull(val)]
        return int(len(set(values)) == 1)
    
    agreement_6 = df[tag_cols].apply(percent_agreement, axis=1).mean()
    agreement_3 = df[[col + '_3class' for col in tag_cols]].apply(percent_agreement, axis=1).mean()

    results.append({
        "pilot": pilot_name,
        'KA_6class': round(alpha_6class, 4),
        'KA_3class': round(alpha_3class, 4),
        'ACC_6class': round(agreement_6 * 100, 1),
        'ACC_3class': round(agreement_3 * 100, 1)
    })

summary = pd.DataFrame(results)
summary = summary.sort_values(by='pilot').reset_index(drop=True)

In [4]:
summary

,pilot,KA_6class,KA_3class,ACC_6class,ACC_3class
0,pilot1,0.2566,0.2573,20.0,50.0
1,pilot2,0.3316,0.3899,38.0,50.0
2,pilot3,0.3715,0.3870,34.0,44.0
3,pilot4,0.4754,0.5012,46.0,55.0
4,pilot5,0.3060,0.3377,48.0,58.0
5,pilot6,0.4363,0.3749,39.0,47.0


In [ ]:
summary.to_csv("../Tables/Pilot_evaluation.csv", index=False, encoding='utf-8')